# Раздел 2: установка, развёртывание и работа с клиентами

Рабочий код ко всем техникам из этого раздела: запускайте ячейки по порядку и смотрите на
результат — заполнять здесь нечего, весь код уже готов.

**Соответствие урокам на Stepik:**

| Пример в этом ноутбуке | Урок на Stepik |
|---|---|
| Режимы развёртывания | «Развертывание Qdrant» |
| Три режима клиента (`url=`/`:memory:`/`path=`) | «Подключение клиента» |
| `AsyncQdrantClient` | «Подключение клиента» |
| REST vs gRPC | «Подключение клиента» |
| Дашборд | «Дашборд» |
| Частые ошибки | «Популярные ошибки» |

**Общая коллекция через volume.** Этот ноутбук подключается к тому же контейнеру
`qdrant_lecture` (volume `qdrant_storage`) и той же коллекции `exercise1_books`, что и
остальная практика курса — если вы уже её проходили, данные просто переиспользуются; если нет,
ячейка настройки соберёт коллекцию заново из `books_dataset.json` (лежит на уровень выше, в
`code/`). Модель эмбеддингов — та же `intfloat/multilingual-e5-small`.

In [ ]:
%pip install qdrant-client sentence-transformers nest_asyncio ipykernel --quiet


In [ ]:
import json
import subprocess
import shutil
import time

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient, AsyncQdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print("OK: библиотеки импортированы")


### Docker: тот же сервер, что и в остальной практике

Коллекции с префиксом `exercise1_`, поэтому спокойно переиспользуем контейнер `qdrant_lecture`.

In [ ]:
CONTAINER_NAME = "qdrant_lecture"


def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip(), result.stderr.strip(), result.returncode


sh("docker volume create qdrant_storage")

_, _, exists = sh("docker inspect {name}".format(name=CONTAINER_NAME))
if exists != 0:
    out, err, _ = sh(
        "docker run -d --name {name} -p 6333:6333 -p 6334:6334 "
        "-v qdrant_storage:/qdrant/storage qdrant/qdrant".format(name=CONTAINER_NAME)
    )
else:
    out, err, _ = sh("docker start {name}".format(name=CONTAINER_NAME))
print(out or err)

client = QdrantClient(url="http://localhost:6333")
print(client.get_collections())


In [ ]:
model = SentenceTransformer("intfloat/multilingual-e5-small")
EMBED_DIM = model.get_embedding_dimension()


def embed_passages(texts):
    return model.encode(["passage: " + t for t in texts], normalize_embeddings=True).tolist()


def embed_query(text):
    return model.encode("query: " + text, normalize_embeddings=True).tolist()


print("Размерность эмбеддинга:", EMBED_DIM)


In [ ]:
with open("../books_dataset.json", encoding="utf-8") as f:
    BOOKS = json.load(f)

BOOK_BY_ID = {b["id"]: b for b in BOOKS}
PREFIX = "exercise1_"
BOOKS_COLLECTION = PREFIX + "books"

# если коллекция уже собрана раньше - просто переиспользуем её; если нет (этот
# ноутбук запущен первым) - соберём с нуля тем же способом
if client.collection_exists(BOOKS_COLLECTION):
    print("BOOKS_COLLECTION уже существует, точек:", client.count(BOOKS_COLLECTION).count)
else:
    client.create_collection(
        collection_name=BOOKS_COLLECTION,
        vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    )
    texts = [b["title"] + ". " + b["text"] for b in BOOKS]
    vectors = embed_passages(texts)
    points = [
        PointStruct(id=b["id"], vector=vec, payload={k: v for k, v in b.items() if k != "id"})
        for b, vec in zip(BOOKS, vectors)
    ]
    BATCH = 16
    for i in range(0, len(points), BATCH):
        client.upsert(collection_name=BOOKS_COLLECTION, points=points[i:i + BATCH])
    print("BOOKS_COLLECTION собрана заново, точек:", client.count(BOOKS_COLLECTION).count)


---
## Пример — режимы развёртывания

Docker уже поднят выше — это самый частый способ для локальной разработки. Ниже — как выглядят
два других распространённых варианта (`docker-compose.yml` и Qdrant Cloud), просто для справки:
второй сервер поднимать здесь незачем, а Cloud требует собственного аккаунта, поэтому эти два
блока не подключаются к реальному серверу, а просто печатают готовый код для копирования.

In [ ]:
compose_yaml = """
services:
  qdrant:
    image: qdrant/qdrant
    ports:
      - "6333:6333"
      - "6334:6334"
    volumes:
      - qdrant_storage:/qdrant/storage
volumes:
  qdrant_storage:
"""
print(compose_yaml)

cloud_snippet = '''
from qdrant_client import QdrantClient

client = QdrantClient(
    url="https://xxxxx-xxxx-xxxx.cloud.qdrant.io",
    api_key="ВАШ_API_КЛЮЧ",
)
print(client.get_collections())
'''
print(cloud_snippet)


---
## Пример — три режима клиента

`url=` — то, что мы использовали до сих пор (отдельный сервер). `path=` и `:memory:` — два
локальных режима без сервера вообще: `path=` пишет на диск (переживает перезапуск процесса),
`:memory:` живёт только пока жив сам Python-процесс. Это два отдельных, изолированных
хранилища — они не имеют отношения к `qdrant_lecture`/`exercise1_books` и специально не делят
с ними данные, поэтому здесь используются игрушечные точки, а не книги.

In [ ]:
# режим url= (уже знаком) - просто напоминание, что мы и так всё время в нём работаем
print("url= :", client.count(BOOKS_COLLECTION).count, "точек в", BOOKS_COLLECTION)

# режим path= - локальное хранилище на диске, свой процесс, ничего общего с Docker-сервером
shutil.rmtree("./tmp_local_qdrant", ignore_errors=True)  # чтобы ячейку можно было перезапускать
path_client = QdrantClient(path="./tmp_local_qdrant")
path_client.create_collection("demo", vectors_config=VectorParams(size=4, distance=Distance.COSINE))
path_client.upsert("demo", points=[PointStruct(id=1, vector=[0.1, 0.2, 0.3, 0.4], payload={})])
print("path=:", path_client.count("demo").count, "точка, сохранена в ./tmp_local_qdrant")
path_client.close()  # обязательно закрыть - иначе следующий клиент не сможет открыть тот же путь

# режим :memory: - живёт только в оперативной памяти текущего процесса
memory_client = QdrantClient(":memory:")
memory_client.create_collection("demo", vectors_config=VectorParams(size=4, distance=Distance.COSINE))
memory_client.upsert("demo", points=[PointStruct(id=1, vector=[0.1, 0.2, 0.3, 0.4], payload={})])
print(":memory:", memory_client.count("demo").count, "точка, исчезнет с концом процесса")


---
## Пример — `AsyncQdrantClient`

Тот же клиент, но с `async`/`await` — полезно внутри асинхронного веб-фреймворка (FastAPI,
aiohttp), чтобы не блокировать event loop на время сетевого запроса к Qdrant.
`nest_asyncio.apply()` нужен только из-за того, что у самого Jupyter уже есть свой запущенный
event loop — в обычном `.py`-скрипте достаточно одного `asyncio.run(...)`.

In [ ]:
import asyncio
import nest_asyncio

nest_asyncio.apply()


async def demo_async_search():
    async_client = AsyncQdrantClient(url="http://localhost:6333")
    hits = await async_client.query_points(
        BOOKS_COLLECTION,
        query=embed_query("одинокий герой и его трагическая судьба"),
        limit=3,
    )
    for h in hits.points:
        print(h.id, round(h.score, 3), h.payload["title"])
    await async_client.close()


asyncio.run(demo_async_search())


---
## Пример — REST vs gRPC

Тот же сервер, тот же запрос — просто другой протокол (`prefer_grpc=True`, порт `6334`,
который мы уже опубликовали при запуске контейнера). gRPC использует бинарный протокол
поверх HTTP/2 вместо JSON поверх HTTP/1.1 — на одиночных запросах разница на локальном
Docker обычно небольшая, но накапливается на высокой частоте запросов.

In [ ]:
grpc_client = QdrantClient(url="http://localhost:6333", prefer_grpc=True, grpc_port=6334)

qv = embed_query("одинокий герой и его трагическая судьба")

rest_hits = client.query_points(BOOKS_COLLECTION, query=qv, limit=3).points
grpc_hits = grpc_client.query_points(BOOKS_COLLECTION, query=qv, limit=3).points
print("REST ids:", [h.id for h in rest_hits])
print("gRPC ids:", [h.id for h in grpc_hits])
print("совпадают:", [h.id for h in rest_hits] == [h.id for h in grpc_hits])

N = 20
t0 = time.perf_counter()
for _ in range(N):
    client.query_points(BOOKS_COLLECTION, query=qv, limit=3)
rest_time = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(N):
    grpc_client.query_points(BOOKS_COLLECTION, query=qv, limit=3)
grpc_time = time.perf_counter() - t0

print("REST: {:.3f} c на {} запросов ({:.1f} мс/запрос)".format(rest_time, N, 1000 * rest_time / N))
print("gRPC: {:.3f} c на {} запросов ({:.1f} мс/запрос)".format(grpc_time, N, 1000 * grpc_time / N))


---
## Дашборд

Никакого кода — просто откройте **http://localhost:6333/dashboard** в браузере, пока
контейнер `qdrant_lecture` запущен. Там можно посмотреть содержимое `exercise1_books`
глазами (payload, векторы), выполнить произвольный поиск через визуальный редактор запроса
и проверить использование памяти — то же самое, что мы всю практику делали кодом, но
без единой строчки Python.

---
## Частые ошибки

Код для всего этого уже был выше — здесь просто чек-лист, за что реально зацепиться на практике:

- **Забыли volume** — `docker run` без `-v qdrant_storage:/qdrant/storage` держит данные только
  внутри контейнера; `docker rm` — и всё пропало. Volume выше был с самого первого запуска.
- **Перепутали порты** — `6333` (REST + дашборд) и `6334` (gRPC) — это два разных порта одного
  сервера, не дублирование одного и того же.
- **Открытый доступ без `api_key`** — без переменной `QDRANT__SERVICE__API_KEY` любой, кто
  достучится до порта, может читать и писать в коллекции (см. раздел «Продакшен» практики).
- **Несовпадение версий клиента и сервера** — `pip install qdrant-client` тянет последнюю
  версию, а сервер может быть старее (или наоборот); при странных ошибках сериализации в
  первую очередь сверьте версии.

---
## Итоги

`url=` (сервер) / `:memory:` / `path=` — три режима клиента без сервера и с ним;
`AsyncQdrantClient` — та же логика, но с `await`, для асинхронных фреймворков;
`prefer_grpc=True` — тот же сервер, другой протокол. Дашборд и чек-лист типичных
ошибок закрывают эксплуатационную часть Раздела 2 — дальше в практике коллекция
`exercise1_books`, собранная здесь (или уже существовавшая), используется без изменений.